![cyber_photo](cyber_photo.jpg)

Cyber threats are a growing concern for organizations worldwide. These threats take many forms, including malware, phishing, and denial-of-service (DOS) attacks, compromising sensitive information and disrupting operations. The increasing sophistication and frequency of these attacks make it imperative for organizations to adopt advanced security measures. Traditional threat detection methods often fall short due to their inability to adapt to new and evolving threats. This is where deep learning models come into play.

Deep learning models can analyze vast amounts of data and identify patterns that may not be immediately obvious to human analysts. By leveraging these models, organizations can proactively detect and mitigate cyber threats, safeguarding their sensitive information and ensuring operational continuity.

As a cybersecurity analyst, you identify and mitigate these threats. In this project, you will design and implement a deep learning model to detect cyber threats. The BETH dataset simulates real-world logs, providing a rich source of information for training and testing your model. The data has already undergone preprocessing, and we have a target label, `sus_label`, indicating whether an event is malicious (1) or benign (0).

By successfully developing this model, you will contribute to enhancing cybersecurity measures and protecting organizations from potentially devastating cyber attacks.


### The Data

| Column     | Description              |
|------------|--------------------------|
|`processId`|The unique identifier for the process that generated the event - int64 |
|`threadId`|ID for the thread spawning the log - int64|
|`parentProcessId`|Label for the process spawning this log - int64|
|`userId`|ID of user spawning the log|Numerical - int64|
|`mountNamespace`|Mounting restrictions the process log works within - int64|
|`argsNum`|Number of arguments passed to the event - int64|
|`returnValue`|Value returned from the event log (usually 0) - int64|
|`sus_label`|Binary label as suspicous event (1 is suspicious, 0 is not) - int64|

More information on the dataset: [BETH dataset](accreditation.md)

In [38]:
# Import required libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torchmetrics import Accuracy
# from sklearn.metrics import accuracy_score  # uncomment to use sklearn

In [39]:
# Load preprocessed data
train_df = pd.read_csv('labelled_train.csv')
test_df = pd.read_csv('labelled_test.csv')
val_df = pd.read_csv('labelled_validation.csv')

# View the first 5 rows of training set
train_df.head()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,7337,1,100,4026532231,5,0,1
1,381,7337,1,100,4026532231,1,0,1
2,381,7337,1,100,4026532231,0,0,1
3,7347,7347,7341,0,4026531840,2,-2,1
4,7347,7347,7341,0,4026531840,4,0,1


In [40]:
train_df.isna().sum()

processId          0
threadId           0
parentProcessId    0
userId             0
mountNamespace     0
argsNum            0
returnValue        0
sus_label          0
dtype: int64

### Data preparation

In [41]:
# Start coding here
# Use as many cells as you need

## split training data
X_train_feature = train_df.iloc[:, :-1]
y_train_label = train_df.iloc[:, -1]

X_train = StandardScaler().fit_transform(X_train_feature)
y_train = y_train_label.to_numpy()

## split testing data
X_test_feature = test_df.iloc[:, :-1]
y_test_label = test_df.iloc[:, -1]

X_test = StandardScaler().fit_transform(X_test_feature)
y_test = y_test_label.to_numpy()

## split validation data
X_val_feature = val_df.iloc[:, :-1]
y_val_label = val_df.iloc[:, -1]

X_val = StandardScaler().fit_transform(X_val_feature)
y_val = y_val_label.to_numpy()

print(X_train)
print(y_train)

[[-3.301279    0.26676142 -0.84909243 ...  1.78399123  1.73607956
  -0.0549941 ]
 [-3.301279    0.26676142 -0.84909243 ...  1.78399123 -1.2469795
  -0.0549941 ]
 [-3.301279    0.26676142 -0.84909243 ...  1.78399123 -1.99274427
  -0.0549941 ]
 ...
 [ 0.23564256  0.23423802  2.35867208 ... -0.58710151 -1.99274427
  -0.0549941 ]
 [ 0.23615568  0.23475427 -0.84909243 ... -0.58710151 -1.99274427
  -0.0549941 ]
 [ 0.15046499  0.14854145 -0.84909243 ... -0.58710151 -1.2469795
  -0.0549941 ]]
[1 1 1 ... 0 0 0]


In [42]:
X_train_ts = torch.tensor(X_train, dtype=torch.float32)
y_train_ts = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_ts = torch.tensor(X_test, dtype=torch.float32)
y_test_ts = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

X_val_ts = torch.tensor(X_val, dtype=torch.float32)
y_val_ts = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# ## Second choice

# dataset_train = TensorDataset(
#     X_train_ts,
#     y_train_ts
# )
# dataset_test = TensorDataset(
#     X_test_ts,
#     y_test_ts
# )
# dataset_val = TensorDataset(
#     X_val_ts,
#     y_val_ts
# )

# ## Access the 1st row of dataset
# feature_sample, label_sample = dataset_train[0]

# print('feture of the 1st row: ', feature_sample)
# print('label of the 1st row: ', label_sample)

In [43]:
# ## Second choice
# ## batch size is the number of data in each batch

# batch_size = 64
# shuffle = True

# dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=shuffle)
# dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=shuffle)
# dataloader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=shuffle)

# ## Access the 1st row of dataset
# ## next(iter(_)) is like when you access the list as list_sample[n] where n is starts from 0

# feature_sample, label_sample = next(iter(dataloader_train))

# print('feture of the 1st row: ', feature_sample)
# print('label of the 1st row: ', label_sample)

### Define Neural Network Model

In [44]:
model = nn.Sequential(
    nn.Linear(X_train_ts.shape[1], 12),
    nn.Linear(12, 10),
    nn.ReLU(),
    nn.Linear(10, 5),
    nn.ReLU(),
    nn.Linear(5, 1),
    nn.Sigmoid()
)

## calculate total number of the parameters
total_param = 0
for n_params in model.parameters():
    total_param += n_params.numel()
print('Total Parameters of the model : ', total_param)

## create the loss & optimizer

## CrossEntropyLoss is used with Classification
## MSELoss is used with Regression & No activation func

critereon = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3, weight_decay=1e-4)

Total Parameters of the model :  287


### Train model

In [45]:
num_epoch = 10

## set the model to training mode, evaluation mode : model.eval()
model.train()

## Training loop
## 1st choice

for epoch in range(num_epoch):
    
    # Clear the gradients
    optimizer.zero_grad()

    # Forward pass: compute the model output
    outputs = model(X_train_ts)

    # Compute the loss
    loss = critereon(outputs, y_train_ts)

    # Backward pass: compute the gradients
    loss.backward()

    # Update the model parameters
    optimizer.step()

## Second choice

# for epoch in range(num_epoch):
#     for feature, label in dataloader_train:
        
#         ## reset gradients
#         optimizer.zero_grad()

#         ## forward pass
#         outputs = model(feature)

#         ## compute the loss
#         loss = critereon(outputs, label)
        
#         ## perform backward pass ( compute gradients )
#         loss.backward()
        
#         ## update the model parameters
#         optimizer.step()

In [49]:
# Set the model to evaluation mode
model.eval()

with torch.no_grad():
    y_pred_train = model(X_train_ts).round()
    y_pred_test = model(X_test_ts).round()
    y_pred_val = model(X_val_ts).round()

accuracy = Accuracy(task="binary")

train_acc = accuracy(y_pred_train, y_train_ts)
test_acc = accuracy(y_pred_test, y_test_ts)
val_accuracy = accuracy(y_pred_val, y_val_ts)

## convert to int/float

train_acc = train_acc.item()
test_acc = test_acc.item()
val_accuracy = val_accuracy.item()

In [50]:
print("Training accuracy: {0}".format(train_acc))
print("Testing accuracy: {0}".format(test_acc))
print("Validation accuracy: {0}".format(val_accuracy))

Training accuracy: 0.0007416686858050525
Testing accuracy: 0.9073383212089539
Validation accuracy: 0.004159456584602594
